<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/proteomics-multiomics/lessons/P07-L01-search-engine-and-fdr/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/proteomics-multiomics/lessons/P07-L01-search-engine-and-fdr/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/proteomics-multiomics/lessons/P07-L01-search-engine-and-fdr/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/proteomics-multiomics/lessons/P07-L01-search-engine-and-fdr/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P07-L01 · From spectra to protein counts: a search engine, target-decoy FDR and protein inference from scratch

**You will build:** a small tandem-mass-spectrometry search engine. It digests a proteome into
peptides, matches fragment ions in each spectrum against every candidate peptide, and reports a
best hit per spectrum. You will then control the false discovery rate with a decoy database, roll
the surviving peptides up to proteins under the parsimony rule, and **measure** how many of the
proteins you report are not in the sample at all, at the same nominal threshold, when error is
controlled on peptides and when it is controlled on proteins.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** `T00-L01-the-8gb-track` (the profiler and the tier gate, and the house
shape of a lesson: stubs, public checks, an autograded rubric). Python and numpy indexing. No
biology beyond "proteins are chains of amino acids and a mass spectrometer weighs their pieces".

By the end you will be able to:

1. Implement a tryptic digest, peptide and b/y fragment-ion masses, and a decoy search space in
   which a peptide shared with a target counts as a target.
2. Implement a hyperscore fragment-matching search that retrieves candidates by precursor-mass
   window and resolves an exact target-decoy tie against the target.
3. Implement target-decoy q-values that treat tied scores as one group, and compare the
   estimate with the true false discovery proportion on spectra whose peptides are known.
4. Implement protein inference by parsimony and protein-level target-decoy scoring.
5. Measure how many reported proteins are absent from the sample when error is controlled on
   peptides and when it is controlled on proteins, at the same promised rate, and explain why the
   two differ.

**The data is synthetic, and the simulator is in this notebook.** The course catalogue's lab
for this module reprocesses a public PRIDE submission, which means downloading raw instrument
files, and this lesson downloads nothing. More importantly, a real submission cannot tell you
which of your protein identifications are wrong: there is no answer key. A simulated sample
has one, and a false discovery rate is a claim about an answer key. `meta.yaml` declares the
data as synthetic, and the programme's dataset ledger (`programmes/proteomics-multiomics/README.md`)
records the terms of the public archive the later modules will use, before any extract of it is
committed. What transfers to a real search is the method. Nothing you conclude about these
particular counts describes any real sample, instrument or search engine.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`

**This lesson assumes you can already:**

- write Python functions over strings, dicts and sets, such as splitting a string at a rule and building a reverse index from values to keys
- sort a numpy array and compute running counts and running ratios down the sorted order
- tell a false positive from a false negative, and a rate from a count

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [A survey of computational methods and error rate estimation procedures for peptide and protein identification in shotgun proteomics](https://pmc.ncbi.nlm.nih.gov/articles/PMC2956504) · Alexey I. Nesvizhskii (J Proteomics 2010, PubMed Central) · *free* — section 5 on target-decoy error rates for spectrum matches and section 7 on protein inference, the two halves of the lesson's FDR and parsimony exercises
- [A Scalable Approach for Protein False Discovery Rate Estimation in Large Proteomic Data Sets](https://pmc.ncbi.nlm.nih.gov/articles/PMC4563723) · Savitski, Wilhelm, Hahne, Kuster and Bantscheff (Mol Cell Proteomics 2015, PubMed Central) · *free* — why the classic target-decoy protein FDR over-predicts false proteins in very large data sets, and the 'picked' remedy that pairs each protein's target and decoy
- [Peptide fragmentation: b and y ions](https://www.matrixscience.com/help/fragmentation_help.html) · Matrix Science (Mascot help) · *free* — how b and y ion masses are built from residue masses, the arithmetic of the fragment-ion exercise
- [PeptideCutter: available enzymes](https://web.expasy.org/peptide_cutter/peptidecutter_enzymes.html) · SIB Swiss Institute of Bioinformatics (Expasy) · *free* — the trypsin cleavage rule (after K or R, not before P) that the digest exercise implements, with its documented exceptions
- [Bioinformatics Algorithms: An Active Learning Approach](https://www.bioinformaticsalgorithms.org/) · Phillip Compeau and Pavel Pevzner (Active Learning Publishers) · *paid* — chapter 4, How Do We Sequence Antibiotics?: inferring a peptide from its mass spectrum, from brute force to the spectral convolution; a gentler on-ramp to the search engine

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import heapq   # not needed unless you want it: a priority queue, for exercise 9
import io
import math
import os
import signal
import sys
import threading
import time
import traceback
from typing import Any, Callable, NamedTuple

import numpy as np

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· matplotlib", matplotlib.__version__)

# The world. Every number that shapes it is here, so that nothing later is a hidden knob.
SEED = 20260929
N_BASE, N_PARALOG, N_FRAGMENT, N_SYMMETRIC = 1280, 120, 75, 3   # kinds of protein, see section 3
N_PRESENT = 150         # proteins actually in the sample; the other proteins are only in the database
N_FOREIGN = 1000        # spectra from peptides that are NOT in the database (see section 3)

# The search settings, fixed for the whole lesson so that only the error control varies.
MISSED = 1              # missed cleavages allowed
MIN_LEN, MAX_LEN = 7, 30
PRECURSOR_PPM = 20.0    # precursor mass tolerance, parts per million of the observed mass
FRAGMENT_TOL = 0.5      # fragment m/z tolerance in Da: a low-resolution instrument
ALPHA = 0.01            # the conventional one-percent line, set here rather than typed in prose
DECOY_PREFIX = "DECOY_"

# Monoisotopic masses: what the most abundant isotope of each element gives, which is what a
# search engine matches against. Residue masses are the amino acids as they sit in a chain (each
# has lost the water that joined it to its neighbours); see claims.yaml for the sources.
PROTON = 1.0072764666    # mass of a proton, in daltons
WATER = 18.0105646838    # one water: the two ends of an intact peptide
RESIDUE_MASS = {
    "G": 57.021464, "A": 71.037114, "S": 87.032028, "P": 97.052764, "V": 99.068414,
    "T": 101.047679, "C": 103.009185, "L": 113.084064, "I": 113.084064, "N": 114.042927,
    "D": 115.026943, "Q": 128.058578, "K": 128.094963, "E": 129.042593, "M": 131.040485,
    "H": 137.058912, "F": 147.068414, "R": 156.101111, "Y": 163.06332, "W": 186.079313,
}
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"


class Spectrum(NamedTuple):
    """One tandem mass spectrum: what the instrument recorded, and nothing else.

    `mz` is sorted ascending; `intensity` is scaled so the tallest peak is 100. The peptide that
    produced it is NOT here: the search engine has to work that out.
    """
    scan: int
    precursor_mz: float
    charge: int
    mz: np.ndarray
    intensity: np.ndarray


def neutral_mass(spectrum: Spectrum) -> float:
    """Mass of the peptide the precursor ion came from: charge times (m/z minus a proton).

    Given to you. A charge-z ion carries z protons that are not part of the peptide.
    """
    return spectrum.charge * (spectrum.precursor_mz - PROTON)


def _show(fig: "matplotlib.figure.Figure") -> None:
    """Display a figure in Jupyter, or close it cleanly in a headless script run."""
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("digest",),
    "exercise 2": ("peptide_mass", "fragment_ions"),
    "exercise 3": ("build_peptide_map",),
    "exercise 4": ("make_decoys", "merge_search_space"),
    "exercise 5": ("match_peaks", "hyperscore"),
    "exercise 6": ("search_spectra",),
    "exercise 7": ("q_values", "accepted"),
    "exercise 8": ("best_per_peptide",),
    "exercise 9": ("protein_groups",),
    "exercise 10": ("protein_results",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run
# A demo that runs the pipeline up to exercise k needs every exercise up to k to have passed.
_UPTO_6 = tuple(f"exercise {i}" for i in range(1, 7))
_UPTO_7 = tuple(f"exercise {i}" for i in range(1, 8))
_UPTO_8 = tuple(f"exercise {i}" for i in range(1, 9))
_UPTO_9 = tuple(f"exercise {i}" for i in range(1, 10))
_UPTO_10 = tuple(f"exercise {i}" for i in range(1, 11))


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (build_peptide_map)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


@contextlib.contextmanager
def _time_limit(seconds: int):
    """Turn a run that never ends into a failed check instead of a frozen notebook.

    Uses the operating system's alarm, so it works where that exists (Linux, macOS, Colab) and is
    skipped, harmlessly, where it does not. An alarm that something else had already set (a grader,
    a time limit around the whole run) is put back afterwards with the time it had left and its
    repeat interval, and if it was due sooner, its deadline wins and its own handler runs.
    """
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    message = (f"still running after {seconds} s. Look for a loop whose condition never changes, or for "
               "code that is right but far too slow for the whole world: comparing every ion with every "
               "peak, or every spectrum with every peptide, instead of only the few near it")
    started = time.monotonic()
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False,
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"] = owner                               # the frame running the `with`

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in this guard's own
        while frame is not None and frame is not state["owner"]:     # set-up, exit or clean-up (or
            if (frame is state["me"] or frame.f_code in (_fire.__code__, _arm.__code__)    # in code
                    or (frame.f_back is state["owner"] and frame.f_globals is vars(contextlib))):
                return True                              # run from contextlib's __exit__ under the `with`)?
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        if state["next"] is not None and now >= state["next"] - 0.005:
            state["fired"] = True
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            raise _OutOfTime(message, state)
        _arm(now)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        raise AssertionError(message) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        raise AssertionError(message)


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`, and
    any earlier demo whose results it uses: until each has passed, the demo says which one it is
    waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    broken = [name for name in needs if name not in _EXERCISES and _STATUS.get(name) != "passed"]
    if broken:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — the {' and '.join(repr(b) for b in broken)} cell above did not "
              "finish; its message says why.")
        return
    try:
        with _time_limit(20 if label in _EXERCISES else 120):   # a check, or a demo on the whole world
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. What a search engine is, in one paragraph

A mass spectrometer does not read sequences. It weighs. Proteins are cut into peptides by an
enzyme; each peptide is weighed (the **precursor** mass), broken, and its pieces are weighed
again (the **fragment** spectrum). The raw material of this whole lesson is that one spectrum:
a list of peaks, each a mass-to-charge ratio and an intensity.

A search engine answers one question per spectrum: *which peptide, from a list of candidates,
best explains this pattern of peaks?* It does so by writing down, for each candidate whose
weight matches the precursor, the fragment masses that peptide *would* produce, and counting
how many of them turn up in the spectrum. The candidate that explains most, wins.

Three facts about that procedure carry the rest of the lesson:

- **It always answers.** A spectrum of something that is not in your list still gets a best
  candidate. That answer is wrong, and nothing in the spectrum says so.
- **A score alone cannot say how wrong the answers are.** You need a way to estimate what
  fraction of accepted answers is wrong. Section 9 builds one.
- **Peptides are not proteins.** A protein count is derived from peptide hits by an inference
  step with its own error, and section 13 measures how large it is.

One scoping note. The spectra here are of the *data-dependent* kind: one selected precursor per
spectrum, so one peptide to find. Data-independent acquisition fragments every peptide in a wide
mass window together, so a spectrum has several answers, and its analysis is peptide-centric
(`claims.yaml`). This search does not do that. The error-control and protein-inference steps that
follow are written to take only scores and peptide-to-protein maps.

Before any of that, the chemistry. Exercises 1 and 2 are the two things every search engine
does first: cut a protein into the peptides trypsin would produce, and compute the masses of
a peptide and of the fragments it breaks into.

In [ ]:
print(f"enzyme        trypsin, up to {MISSED} missed cleavage, peptides of {MIN_LEN} to {MAX_LEN} residues")
print(f"tolerances    precursor {PRECURSOR_PPM:g} ppm, fragments {FRAGMENT_TOL:g} Da")
print("residues (Da) " + "  ".join(f"{k} {v:.3f}" for k, v in sorted(RESIDUE_MASS.items())[:10]))
print("              " + "  ".join(f"{k} {v:.3f}" for k, v in sorted(RESIDUE_MASS.items())[10:]))
print("note that I and L weigh exactly the same: no mass spectrometer can tell them apart.")

## 2. Exercise 1 — `digest()`

Trypsin, the enzyme almost every proteomics lab uses, cuts a protein chain **after** each
lysine (`K`) or arginine (`R`), except that it does not cut when the next residue is a proline
(`P`). That is the simplified rule this lesson uses; real trypsin has further exceptions,
listed in `claims.yaml` with the source.

The cutting is not perfect, and a peptide with an uncut site inside it (a *missed cleavage*)
is common enough that search engines allow for it. So `digest` returns every stretch of the
chain that starts at a cut site and ends at a cut site at most `missed_cleavages` sites
further on, if its length is within the limits.

<details><summary>💡 Hint 1 — what to think about</summary>

First find the places where the chain is cut, as positions between residues, and remember that
the very start and the very end of the chain are boundaries too. A `K` or `R` in the last
position has nothing after it, so it is not followed by a proline. What does that leave you
with: a list of boundary positions, and peptides as spans between one boundary and a later one?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Collect the boundary positions in order. For each starting boundary, join it to each of the
next `missed_cleavages + 1` boundaries in turn, slice the chain, and keep the span when its
length is within the limits, both limits included. Report each distinct peptide once, in the
order first met. Reject a `missed_cleavages` that is not a whole number of at least zero with
`ValueError` before anything else.
</details>

In [ ]:
def digest(sequence: str, missed_cleavages: int = 1, min_len: int = MIN_LEN,
           max_len: int = MAX_LEN) -> list[str]:
    """Tryptic peptides of one protein sequence.

    Trypsin cuts after `K` or `R` unless the next residue is `P`. A peptide runs from one cut
    site (or the start of the chain) to a later one (or the end), spanning at most
    `missed_cleavages` interior sites. Keep it if `min_len <= len(peptide) <= max_len`.

    Peptides are ordered by start, then by end, and each distinct peptide appears once. Raise
    `ValueError` if `missed_cleavages` is not a whole number of at least 0.

    Example:
        >>> digest("AAAAAAKPAAAAAAKAAAAAAAR", missed_cleavages=0, min_len=6, max_len=30)
        ['AAAAAAKPAAAAAAK', 'AAAAAAAR']
        >>> digest("AAAAAAKCCCCCCR", missed_cleavages=1, min_len=6, max_len=30)
        ['AAAAAAK', 'AAAAAAKCCCCCCR', 'CCCCCCR']

    Returns:
        A list of peptide strings.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_digest() -> None:
    got = digest("AAAAAAKPAAAAAAKAAAAAAAR", missed_cleavages=0, min_len=6, max_len=30)
    assert got == ["AAAAAAKPAAAAAAK", "AAAAAAAR"], (
        f"expected ['AAAAAAKPAAAAAAK', 'AAAAAAAR'], got {got!r}. A K or R followed by P is NOT "
        "a cut site: the first K here must stay inside its peptide"
    )
    got = digest("AAAAAAKCCCCCCR", missed_cleavages=1, min_len=6, max_len=30)
    assert got == ["AAAAAAK", "AAAAAAKCCCCCCR", "CCCCCCR"], (
        f"expected ['AAAAAAK', 'AAAAAAKCCCCCCR', 'CCCCCCR'], got {got!r}: with one missed "
        "cleavage allowed, each start joins to the next boundary AND the one after it"
    )
    got = digest("AAAAAAKCCCCCCCR", missed_cleavages=0, min_len=7, max_len=7)
    assert got == ["AAAAAAK"], (
        f"with min_len = max_len = 7 the answer is ['AAAAAAK'] (7 residues; the other peptide has "
        f"8), got {got!r}; both length limits are inclusive"
    )
    assert digest("", 1, 1, 30) == [], "an empty chain has no peptides"
    try:
        digest("AAAAAAK", -1)
    except ValueError:
        pass
    else:
        raise AssertionError("a negative missed_cleavages must raise ValueError")
    print("exercise 1 looks right — digest cuts after K and R, but not before P")

In [ ]:
_try("exercise 1", _check_digest)

## 3. Exercise 2 — `peptide_mass()` and `fragment_ions()`

The residue masses are in the setup cell. A peptide's **neutral mass** is the sum of its
residue masses plus one water: the extra hydrogen on the amine end and the extra hydroxyl on
the acid end.

When a peptide is broken along its backbone, each break yields one piece that keeps the amine
end and one that keeps the acid end. The mass spectrometer sees only pieces that carry a
charge. This lesson uses the two commonest kinds, each with a single charge, one proton:

- **b ions** keep the amine end. b<sub>i</sub> is the sum of the first *i* residue masses plus
  one proton.
- **y ions** keep the acid end. y<sub>i</sub> is the sum of the last *i* residue masses plus
  one water plus one proton.

A peptide of *n* residues has *n − 1* breaks, so *n − 1* b ions and *n − 1* y ions. A real
spectrum also has doubly charged fragments, neutral losses and modified residues. Ignoring
them here keeps the engine small and the errors visible, and it makes this a toy, not a
rival to production engines such as Sage and MSFragger (`claims.yaml`).

<details><summary>💡 Hint 1 — what to think about</summary>

A b ion and the y ion that complements it come from the same break. Together they hold every
residue once, plus the two ends, plus two protons. Use that as a cross-check on your own
arrays. Also: what should happen to a peptide containing a letter that is not an amino acid?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Look each residue up in `RESIDUE_MASS`; raise `ValueError` naming the offending letter, and
for an empty peptide, rather than letting a `KeyError` or a silent zero through. For the b
ions, accumulate from the amine end; for the y ions, start from a water and accumulate from
the acid end; add the proton to each. Return the two arrays each in order of increasing *i*.
</details>

In [ ]:
def peptide_mass(peptide: str) -> float:
    """Neutral monoisotopic mass of a peptide: its residues plus one water.

    Raise `ValueError` for an empty peptide or a letter that is not in `RESIDUE_MASS`; the
    message names the letter.

    Example:
        >>> round(peptide_mass("GA"), 6)      # 57.021464 + 71.037114 + one water
        146.069143

    Returns:
        The mass in daltons, as a plain Python float.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def fragment_ions(peptide: str) -> tuple[np.ndarray, np.ndarray]:
    """Singly charged b and y ion m/z values, in order of increasing ion number.

    b_i = (sum of the first i residue masses) + PROTON, for i = 1 .. n-1.
    y_i = (sum of the last i residue masses) + WATER + PROTON, for i = 1 .. n-1.

    Raise `ValueError` for a peptide of fewer than 2 residues or with an unknown residue.

    Example:
        >>> b, y = fragment_ions("GA")
        >>> b.round(4).tolist(), y.round(4).tolist()
        ([58.0287], [90.055])

    Returns:
        `(b, y)`: two float arrays of length `len(peptide) - 1`; `b[i-1]` is b_i, `y[i-1]` is y_i.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_masses() -> None:
    m = peptide_mass("GA")
    assert isinstance(m, float) and abs(m - 146.0691426838) < 1e-6, (
        f"peptide_mass('GA') should be 57.021464 + 71.037114 + 18.010565 = 146.069143, got "
        f"{m!r}. Forgetting the water gives 128.058578"
    )
    for bad in ("", "GAX", "GA1"):
        try:
            peptide_mass(bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"peptide_mass({bad!r}) must raise ValueError")
    b, y = fragment_ions("GAS")
    assert len(b) == 2 and len(y) == 2, (
        f"a 3-residue peptide has 2 breaks, so 2 b ions and 2 y ions; got {len(b)} and {len(y)}"
    )
    assert abs(b[0] - 58.0287404666) < 1e-6 and abs(b[1] - 129.0658544666) < 1e-6, (
        f"b ions of GAS should be G+proton and G+A+proton = 58.0287, 129.0659; got "
        f"{np.round(b, 4).tolist()}"
    )
    assert abs(y[0] - 106.0498691504) < 1e-6 and abs(y[1] - 177.0869831504) < 1e-6, (
        f"y ions of GAS should be S+water+proton and A+S+water+proton = 106.0499, 177.0870; "
        f"got {np.round(y, 4).tolist()}. y_1 is the LAST residue, not the first"
    )
    whole = peptide_mass("GAS")
    assert np.allclose(b + y[::-1], whole + 2 * PROTON, atol=1e-6), (
        "b_i + y_(n-i) must equal the peptide mass plus two protons: the two pieces of one break "
        "hold every residue exactly once"
    )
    try:
        fragment_ions("G")
    except ValueError:
        pass
    else:
        raise AssertionError("a 1-residue peptide cannot break: fragment_ions must raise ValueError")
    print("exercise 2 looks right — masses and b/y ions agree, and complement each other")

In [ ]:
_try("exercise 2", _check_masses)

## 4. The world you will search: a proteome, a sample, and the spectra it produces

Below is the simulator. Read it: a simulated sample has the one advantage a real one lacks,
an answer key, and that is what lets this lesson *measure* how wrong your results are.

- **A database of proteins**, of four kinds. *Base* proteins have random sequences drawn from an
  invented, roughly protein-like composition. *Paralogs* are copies of a base protein with a few
  residues changed, as gene duplicates are. *Fragments* are truncated copies, so apart from the
  few peptides at their two new ends, every peptide they have, another protein has too.
  *Symmetric* proteins read the same forwards and
  backwards; real proteins have repeats and low-complexity stretches, and here they are planted
  deliberately as a trap for exercise 4.
- **A sample** that contains only some of those proteins. The rest of the database is what a
  wrong hit will land on. Each protein in the sample gives a number of spectra that grows with
  how abundant it is, and every spectrum records one of its peptides.
- **Spectra of peptides that are not in the database at all** (`N_FOREIGN` of them): a peptide
  from a contaminant, an unlisted isoform, a chemically modified form. They are the reason a
  search engine's answers have to be audited, because the engine answers for them too.
- **Spectrum quality that varies.** Each spectrum contains some fraction of its peptide's
  fragment ions, jittered in m/z, among a haze of noise peaks. Poor spectra look like nothing.

The simulator carries **its own copy** of the chemistry, written differently from yours, so a bug
in your digest or your masses cannot corrupt the world you are being tested against. That is also
why this cell comes after exercises 1 and 2, and the cell after it compares your code against
it. The generator is seeded: the same world, on any machine, with nothing downloaded.

In [ ]:
_WEIGHTS = {"A": 8.0, "C": 1.5, "D": 5.5, "E": 6.5, "F": 4.0, "G": 7.0, "H": 2.3, "I": 5.5,
            "K": 5.8, "L": 9.5, "M": 2.2, "N": 4.0, "P": 4.8, "Q": 4.0, "R": 5.5, "S": 6.7,
            "T": 5.5, "V": 6.8, "W": 1.2, "Y": 3.0}   # invented, protein-like; measured from nothing
_CDF = np.cumsum([_WEIGHTS[a] for a in AMINO_ACIDS]) / sum(_WEIGHTS.values())


class World(NamedTuple):
    proteins: dict          # protein id -> sequence: the whole database
    present: frozenset      # ids of the proteins actually in the sample
    spectra: list           # list[Spectrum], in scan order
    truth: list             # per spectrum: the peptide that made it, or None if not in the database


def _draw_sequence(rng: np.random.Generator, n: int) -> str:
    idx = np.minimum(np.searchsorted(_CDF, rng.random(n), side="right"), len(AMINO_ACIDS) - 1)
    return "".join(AMINO_ACIDS[i] for i in idx)


def _sim_digest(sequence: str) -> list[str]:
    """The simulator's own tryptic digest, with the lesson's settings."""
    cuts = [i + 1 for i in range(len(sequence) - 1)
            if sequence[i] in "KR" and sequence[i + 1] != "P"]
    edges = [0] + cuts + [len(sequence)]
    found: dict[str, None] = {}
    for a in range(len(edges) - 1):
        for b in range(a + 1, min(a + 2 + MISSED, len(edges))):
            peptide = sequence[edges[a]:edges[b]]
            if MIN_LEN <= len(peptide) <= MAX_LEN:
                found.setdefault(peptide)
    return list(found)


def _sim_fragments(peptide: str) -> tuple[np.ndarray, np.ndarray]:
    m = np.array([RESIDUE_MASS[a] for a in peptide])
    return np.cumsum(m)[:-1] + PROTON, np.cumsum(m[::-1])[:-1] + WATER + PROTON


def _sim_mass(peptide: str) -> float:
    return WATER + sum(RESIDUE_MASS[a] for a in peptide)


def _sim_spectrum(rng: np.random.Generator, peptide: str) -> Spectrum:
    charge = 2 if rng.random() < 0.6 else 3
    quality = 0.15 + 0.6 * rng.random()          # the chance that any one fragment ion shows up
    b, y = _sim_fragments(peptide)
    ions = np.concatenate([b, y])
    is_y = np.concatenate([np.zeros(len(b), dtype=bool), np.ones(len(y), dtype=bool)])
    shown = rng.random(len(ions)) < quality
    signal_mz = ions[shown] + rng.normal(0.0, 0.12, int(shown.sum()))
    signal_int = np.exp(rng.normal(np.where(is_y[shown], 4.0, 3.6), 0.6))
    mass = _sim_mass(peptide) * (1.0 + rng.normal(0.0, 4e-6))   # a few ppm of measurement error
    n_noise = int(100 + 100 * rng.random())
    noise_mz = 100.0 + (min(2000.0, mass) - 100.0) * rng.random(n_noise)
    noise_int = np.exp(rng.normal(2.6, 0.8, n_noise))
    mz = np.concatenate([signal_mz, noise_mz])
    intensity = np.concatenate([signal_int, noise_int])
    order = np.argsort(mz, kind="stable")
    mz, intensity = mz[order], 100.0 * intensity[order] / intensity.max()
    return Spectrum(0, (mass + charge * PROTON) / charge, charge, mz, intensity)


def generate_world(seed: int = SEED) -> World:
    """Deterministic synthetic proteome, sample and spectra. Same seed, same world."""
    rng = np.random.default_rng(seed)
    proteins: dict[str, str] = {}

    def add(sequence: str) -> None:
        proteins[f"P{len(proteins) + 1:04d}"] = sequence

    for _ in range(N_BASE):
        add(_draw_sequence(rng, int(np.clip(np.exp(rng.normal(5.7, 0.5)), 80, 1200))))
    base_ids = list(proteins)
    for _ in range(N_PARALOG):                    # copy with roughly one residue in seventeen changed
        source = proteins[base_ids[int(rng.random() * len(base_ids))]]
        changed = rng.random(len(source)) < 0.06
        fresh = rng.random(len(source))
        add("".join(AMINO_ACIDS[int(fresh[i] * 20)] if changed[i] else source[i]
                    for i in range(len(source))))
    for _ in range(N_FRAGMENT):                   # a truncated copy: bar its new ends, its parent's peptides
        source = proteins[base_ids[int(rng.random() * len(base_ids))]]
        start = int(rng.random() * 0.3 * len(source))
        add(source[start:len(source) - int(rng.random() * 0.3 * len(source))])
    for _ in range(N_SYMMETRIC):                  # reads the same in both directions
        half = _draw_sequence(rng, 60 + int(rng.random() * 80))
        add(half + half[::-1])

    ids = list(proteins)
    picked = np.argsort(rng.random(len(ids)), kind="stable")[:N_PRESENT]
    present = frozenset(ids[i] for i in picked)

    spectra: list[Spectrum] = []
    truth: list = []
    for i in picked:                              # spectra from proteins that are in the sample
        abundance = float(np.exp(rng.normal(1.0, 1.0)))
        n_spectra = 1 + int(abundance * 2.0)
        candidates = _sim_digest(proteins[ids[i]])
        for _ in range(n_spectra):
            if candidates:
                peptide = candidates[int(rng.random() * len(candidates))]
                spectra.append(_sim_spectrum(rng, peptide))
                truth.append(peptide)
    for _ in range(N_FOREIGN):                    # spectra of peptides the database has never heard of
        length = int(7 + rng.random() * 18)
        peptide = _draw_sequence(rng, length - 1) + ("K" if rng.random() < 0.5 else "R")
        spectra.append(_sim_spectrum(rng, peptide))
        truth.append(None)
    order = np.argsort(rng.random(len(spectra)), kind="stable")   # scan order tells nothing
    return World(proteins, present,
                 [spectra[j]._replace(scan=k + 1) for k, j in enumerate(order)],
                 [truth[j] for j in order])


WORLD = generate_world()
PROTEINS, PRESENT, SPECTRA, TRUTH = WORLD
n_foreign = sum(t is None for t in TRUTH)
print(f"database:  {len(PROTEINS)} proteins, {sum(map(len, PROTEINS.values()))} residues; "
      f"{len(PRESENT)} of them are in the sample")
print(f"spectra:   {len(SPECTRA)} in all; {n_foreign} ({100 * n_foreign / len(SPECTRA):.1f}%) come "
      "from peptides that are not in the database at all")
print(f"           {sum(len(s.mz) for s in SPECTRA) / len(SPECTRA):.0f} peaks per spectrum on average")


def _same_peptide(a: str, b: str) -> bool:
    """True if two peptides are the same to a mass spectrometer: I and L weigh the same."""
    return a.replace("I", "L") == b.replace("I", "L")

Two spectra, one from a peptide that is in the database and one from a peptide that is not.
By eye they are both a forest of sticks. Nothing in a spectrum says whether the answer to
"which peptide?" is in your list.

In [ ]:
_first_known = next(i for i, t in enumerate(TRUTH) if t is not None)
_first_foreign = next(i for i, t in enumerate(TRUTH) if t is None)
_fig, _axes = plt.subplots(1, 2, figsize=(10, 2.6), sharey=True)
for _ax, _i, _title in ((_axes[0], _first_known, "peptide IS in the database"),
                        (_axes[1], _first_foreign, "peptide is NOT in the database")):
    _s = SPECTRA[_i]
    _ax.vlines(_s.mz, 0, _s.intensity, lw=0.7)
    _ax.set_title(f"scan {_s.scan}: {_title}", fontsize=9)
    _ax.set_xlabel("m/z")
_axes[0].set_ylabel("relative intensity")
_fig.tight_layout()
_show(_fig)

In [ ]:
def _check_against_the_simulator() -> None:
    ids = list(PROTEINS)[:300]
    for pid in ids:
        want = _sim_digest(PROTEINS[pid])
        got = digest(PROTEINS[pid], MISSED, MIN_LEN, MAX_LEN)
        assert got == want, (
            f"digest disagrees with the simulator on protein {pid}: yours gives {len(got)} "
            f"peptides, the simulator's {len(want)}. First difference: "
            f"{next((g for g, w in zip(got, want) if g != w), 'a length difference')!r}"
        )
    peptides = sorted({p for pid in ids for p in _sim_digest(PROTEINS[pid])})[:500]
    for p in peptides:
        assert abs(peptide_mass(p) - _sim_mass(p)) < 1e-6, f"peptide_mass({p!r}) disagrees"
        b, y = fragment_ions(p)
        sb, sy = _sim_fragments(p)
        assert np.allclose(b, sb, atol=1e-6) and np.allclose(y, sy, atol=1e-6), (
            f"fragment_ions({p!r}) disagrees with the simulator"
        )
    print(f"your digest agrees with the simulator on {len(ids)} of {len(ids)} proteins")
    print(f"your masses and fragment ions agree on {len(peptides)} of {len(peptides)} peptides")


_try("simulator cross-check", _check_against_the_simulator, needs=("exercise 1", "exercise 2"))

## 5. Exercise 3 — `build_peptide_map()`

A search engine works on peptides, but the question you are ultimately asked is about
proteins. The bridge is a map from each peptide sequence to every protein that contains it.

It is not one-to-one. A peptide can occur in several proteins: paralogs, isoforms, and the
truncated forms in this database. Such a *shared* peptide cannot say which of its proteins was
in the sample, and that ambiguity is the protein inference problem of section 11.

<details><summary>💡 Hint 1 — what to think about</summary>

Which direction do you need to look things up in: from a peptide to its proteins, or the other
way round? What should be stored for a peptide that occurs twice inside one protein: the
protein twice, or once?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Digest every protein with your `digest` and the settings you were given, and record for each
peptide the set of proteins it came from. Store each peptide's proteins as a tuple of ids, sorted,
so that the same peptide always maps to the same tuple.
</details>

In [ ]:
def build_peptide_map(proteins: dict, missed_cleavages: int = MISSED, min_len: int = MIN_LEN,
                      max_len: int = MAX_LEN) -> dict:
    """Map each tryptic peptide to the proteins that contain it.

    `proteins` maps protein id to sequence. The result maps each distinct peptide to a tuple of
    the ids of every protein whose digest contains it, sorted, each id once.

    Example:
        >>> build_peptide_map({"B": "AAAAAAAK", "A": "AAAAAAAKCCCCCCCR"}, 0, 7, 30)
        {'AAAAAAAK': ('A', 'B'), 'CCCCCCCR': ('A',)}

    Returns:
        A dict `{peptide: (protein_id, ...)}`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_build_peptide_map() -> None:
    got = build_peptide_map({"B": "AAAAAAAK", "A": "AAAAAAAKCCCCCCCR"}, 0, 7, 30)
    assert got == {"AAAAAAAK": ("A", "B"), "CCCCCCCR": ("A",)}, (
        f"expected AAAAAAAK shared by ('A', 'B') and CCCCCCCR only in ('A',), got {got!r}. "
        "A peptide that occurs in two proteins must list BOTH, sorted"
    )
    twice = build_peptide_map({"X": "AAAAAAAKAAAAAAAK"}, 0, 7, 30)
    assert twice == {"AAAAAAAK": ("X",)}, (
        f"a peptide occurring twice in one protein lists that protein once, got {twice!r}"
    )
    print("exercise 3 looks right — every peptide knows every protein it could have come from")

In [ ]:
_try("exercise 3", _check_build_peptide_map)

## 6. Exercise 4 — decoys, and the search space

Here is the difficulty. The engine always returns a best candidate, correct or not, and the
score alone does not tell you which. To estimate how many wrong answers are hiding among the
accepted ones, add to the database a set of **decoy** proteins that cannot be in any sample,
and search everything together. A wrong hit is equally likely, by assumption, to land on a
target or on a decoy; so the number of decoy hits above a score tells you about the number of
wrong target hits above it. The method is the target-decoy strategy of Elias and Gygi, and the
assumption is exactly as strong as it sounds. Section 8 tests it.

The simplest decoy of a protein is its **reversed** sequence: same composition, same length,
and almost never a real protein (the second rule below is about the exception). Shuffling is the
common alternative; Elias and Gygi report that other ways of building decoys are similarly
effective once certain considerations are taken into account (`claims.yaml`). Two rules make
reversal work in practice, and both are graded:

- a decoy protein takes the target's id with `DECOY_PREFIX` in front, so a decoy hit can be
  told from a target hit by its id alone;
- if a decoy peptide is also a target peptide (a symmetric protein reversed is itself), the
  peptide **is a target**: the search space lists it under its target proteins only, so no hit
  on it is ever counted as a decoy hit.

<details><summary>💡 Hint 1 — what to think about</summary>

When a decoy peptide has exactly the same sequence as a target peptide, the engine cannot tell
them apart, so what can a hit on it honestly be called? And which of the two possible mistakes
(calling it a decoy, or calling it a target) makes the decoy count too big, and which too small?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

`make_decoys` builds a new dict: prefixed id, reversed sequence, the input left untouched.
`merge_search_space` starts from the target map and adds each decoy peptide the target map does
not contain. It returns a new dict and changes neither input.
</details>

In [ ]:
def make_decoys(proteins: dict) -> dict:
    """Decoy proteins: each target's sequence reversed, under an id with DECOY_PREFIX in front.

    Do not modify `proteins`.

    Example:
        >>> make_decoys({"P1": "ACDK"})
        {'DECOY_P1': 'KDCA'}

    Returns:
        A dict `{DECOY_PREFIX + id: reversed sequence}`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def merge_search_space(target_map: dict, decoy_map: dict) -> dict:
    """Combine a target peptide map and a decoy peptide map into the one space that is searched.

    A peptide present in `target_map` keeps only its target proteins, whatever `decoy_map` says
    about it. A peptide only in `decoy_map` keeps its decoy proteins. Neither input is changed.

    Example:
        >>> t = {"AAAAAAK": ("P1",)}
        >>> d = {"AAAAAAK": ("DECOY_P2",), "KAAAAAA": ("DECOY_P1",)}
        >>> merge_search_space(t, d)
        {'AAAAAAK': ('P1',), 'KAAAAAA': ('DECOY_P1',)}

    Returns:
        A new dict `{peptide: (protein_id, ...)}`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_decoys() -> None:
    src = {"P1": "ACDK", "P2": "MMR"}
    got = make_decoys(src)
    assert got == {"DECOY_P1": "KDCA", "DECOY_P2": "RMM"}, (
        f"expected reversed sequences under DECOY_-prefixed ids, got {got!r}"
    )
    assert src == {"P1": "ACDK", "P2": "MMR"}, "make_decoys must not modify its input"
    t = {"AAAAAAK": ("P1",)}
    d = {"AAAAAAK": ("DECOY_P2",), "KAAAAAA": ("DECOY_P1",)}
    merged = merge_search_space(t, d)
    assert merged == {"AAAAAAK": ("P1",), "KAAAAAA": ("DECOY_P1",)}, (
        f"a peptide found in the targets must list only its target proteins, got {merged!r}. "
        "Listing the decoy protein too would let a decoy hit be counted on a target peptide"
    )
    assert t == {"AAAAAAK": ("P1",)} and len(d) == 2, "merge_search_space must not modify its inputs"
    print("exercise 4 looks right — the decoy database, and the collision rule")

In [ ]:
_try("exercise 4", _check_decoys)

The two given cells below build the search space from the whole database and index it by mass,
so that a spectrum's candidates can be found without looking at every peptide. Nothing in
them is yours to write, but every number they print comes from your code.

In [ ]:
class PeptideIndex(NamedTuple):
    """The search space, sorted by peptide mass so a mass window is one contiguous slice."""
    masses: np.ndarray       # neutral masses, ascending
    peptides: tuple          # peptide sequences, aligned with `masses`
    is_decoy: np.ndarray     # bool, aligned: True if every protein holding the peptide is a decoy


def build_index(space: dict) -> PeptideIndex:
    """Sort a search space by (mass, sequence). Given; uses your `peptide_mass`."""
    rows = sorted((peptide_mass(p), p) for p in space)
    peptides = tuple(p for _, p in rows)
    return PeptideIndex(
        np.array([m for m, _ in rows], dtype=float), peptides,
        np.array([all(pid.startswith(DECOY_PREFIX) for pid in space[p]) for p in peptides],
                 dtype=bool))


_PIPE: dict[str, Any] = {}   # results of the pipeline stages, so later cells can use them


def _build_search_space() -> None:
    target_map = build_peptide_map(PROTEINS)
    decoy_map = build_peptide_map(make_decoys(PROTEINS))
    space = merge_search_space(target_map, decoy_map)
    index = build_index(space)
    dropped = sum(1 for p in decoy_map if p in target_map)
    shared = sum(1 for ids in target_map.values() if len(ids) > 1)
    _PIPE.update(space=space, index=index)
    print(f"target peptides:  {len(target_map)}   ({shared} occur in more than one protein)")
    print(f"decoy peptides:   {len(decoy_map)}   ({dropped} of them are also target peptides "
          "and are treated as targets)")
    print(f"search space:     {len(index.peptides)} peptides, of which {int(index.is_decoy.sum())} "
          "are decoys")


_try("search space", _build_search_space,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 7. Exercise 5 — matching peaks, and the hyperscore

Now the heart of the engine. Given a spectrum and one candidate peptide, how well does the
candidate explain it? Two functions.

`match_peaks` takes a list of theoretical fragment masses and the observed peaks, and reports
how many theoretical ions have an observed peak within a tolerance, and how much intensity
those peaks carry. The rule is fixed: for each theoretical ion look at the **nearest** observed
peak; it is a match if it lies within the tolerance, **the tolerance itself included**. If two
observed peaks are exactly as near, take the one at the lower m/z. Each ion is judged on its own.

`hyperscore` combines the two ion series. With *n*<sub>b</sub> and *n*<sub>y</sub> matched b and y
ions, and *I*<sub>b</sub> and *I*<sub>y</sub> the summed intensity of the peaks they matched, it is

&nbsp;&nbsp;&nbsp;&nbsp;ln((*I*<sub>b</sub> + 1)(*I*<sub>y</sub> + 1)) + ln(*n*<sub>b</sub>!) + ln(*n*<sub>y</sub>!)

the X!Tandem-style hyperscore, in the form the open engine Sage computes it (its source code,
as read on 14 October 2026; but with exact log-factorials, `math.lgamma(n + 1)`, not Stirling's
approximation, so the numbers are not comparable with Sage's; source in `claims.yaml`).
The factorial terms reward concentrated matches: *n* ions matched in one series earn more than
the same *n* split over the two, because ln(*n*!) grows faster than *n*. The cell after this
exercise prints how steeply it grows, and one comparison.

<details><summary>💡 Hint 1 — what to think about</summary>

The observed m/z values are sorted, and that is a gift: for a given theoretical value, the
nearest observed peak is one of the two peaks either side of the place it would be inserted.
Which two, and what if the value sits before the first peak or after the last?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Find the insertion position of each theoretical value among the observed m/z values, compare the
gap to the peak on each side (a tie goes to the lower one), keep the nearest, and count the ions
whose nearest gap is within the tolerance. Raise `ValueError` for a negative tolerance. For
`hyperscore`, split the peptide into its b and y ions, match each series on its own, and combine.
An ion series with no matches contributes a factorial of zero, which is one.
</details>

In [ ]:
def match_peaks(theoretical: np.ndarray, mz: np.ndarray, intensity: np.ndarray,
                tol: float) -> tuple[int, float]:
    """Count theoretical ions that have an observed peak within `tol`, and sum those peaks.

    For each value in `theoretical`, take the NEAREST observed peak (the lower m/z if two are
    equally near). It is a match if its distance is `<= tol`. Each ion is judged independently.
    `mz` is sorted ascending. Raise `ValueError` if `tol` is negative.

    Example:
        >>> mz, inten = np.array([100.5, 199.0, 200.4, 250.0]), np.array([10.0, 20.0, 30.0, 40.0])
        >>> match_peaks(np.array([100.0, 200.0, 300.0]), mz, inten, 0.5)
        (2, 40.0)

    Returns:
        `(number of matched ions, total intensity of their peaks)`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def hyperscore(spectrum: Spectrum, peptide: str, tol: float) -> float:
    """ln((Ib + 1)(Iy + 1)) + ln(nb!) + ln(ny!) for one spectrum and one candidate peptide.

    `nb`, `ny` count matched b and y ions and `Ib`, `Iy` sum the intensities of their peaks,
    from `match_peaks` run on the b ions and the y ions separately. Use `math.lgamma(n + 1)` for
    ln(n!). A candidate with no matches scores 0.0.

    Example:
        >>> s = Spectrum(1, 500.0, 2, np.array([58.0287404666, 106.0498691504]), np.array([50.0, 20.0]))
        >>> round(hyperscore(s, "GAS", 0.01), 4)      # nb = 1, ny = 1: ln(51 * 21) + 0 + 0
        6.9763

    Returns:
        The score, as a plain Python float.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_match_peaks() -> None:
    mz = np.array([100.5, 199.0, 200.4, 250.0])
    inten = np.array([10.0, 20.0, 30.0, 40.0])
    n, total = match_peaks(np.array([100.0, 200.0, 300.0]), mz, inten, 0.5)
    assert n == 2 and abs(total - 40.0) < 1e-9, (
        f"expected 2 matches carrying 10 + 30 = 40 units, got {n} carrying {total!r}. Ion 100.0 "
        "sits exactly one tolerance from the peak at 100.5 and counts; ion 200.0 must pick its "
        "NEAREST peak (200.4), not the first one it meets (199.0); ion 300.0 has nothing near"
    )
    n, total = match_peaks(np.array([100.0]), np.array([99.75, 100.25]), np.array([5.0, 7.0]), 0.5)
    assert n == 1 and abs(total - 5.0) < 1e-9, (
        f"two peaks equally near: take the one at the lower m/z (intensity 5), got {total!r}"
    )
    assert match_peaks(np.array([50.0, 400.0]), mz, inten, 0.5) == (0, 0.0), (
        "ions beyond both ends of the spectrum have no neighbour on one side: no match, no crash"
    )
    assert match_peaks(np.array([100.0]), np.array([]), np.array([]), 0.5) == (0, 0.0), (
        "an empty spectrum matches nothing"
    )
    try:
        match_peaks(np.array([1.0]), mz, inten, -0.1)
    except ValueError:
        pass
    else:
        raise AssertionError("a negative tolerance must raise ValueError")
    print("exercise 5 (match_peaks) looks right")


def _check_hyperscore() -> None:
    s = Spectrum(1, 500.0, 2, np.array([58.0287404666, 106.0498691504, 129.0658544666]),
                 np.array([50.0, 20.0, 30.0]))
    got = hyperscore(s, "GAS", 0.01)
    want = math.log(81.0 * 21.0) + math.log(2.0)
    assert isinstance(got, float) and abs(got - want) < 1e-9, (
        f"peaks sit on b1, b2 and y1 of GAS (nb = 2 with intensity 80, ny = 1 with intensity 20): "
        f"expected ln(81 * 21) + ln(2!) = {want:.6f}, got {got!r}. The two series are matched "
        "and summed SEPARATELY, and the +1 goes on each intensity sum"
    )
    far = Spectrum(2, 500.0, 2, np.array([900.0, 950.0]), np.array([100.0, 90.0]))
    assert abs(hyperscore(far, "GAS", 0.01)) < 1e-12, (
        "a candidate that explains no peak at all scores exactly 0.0 (ln 1 + ln 0! + ln 0!)"
    )
    y_only = Spectrum(3, 500.0, 2, np.array([106.0498691504, 177.0869831504]), np.array([10.0, 30.0]))
    want = math.log(1.0 * 41.0) + math.log(2.0)
    assert abs(hyperscore(y_only, "GAS", 0.01) - want) < 1e-9, (
        f"two y ions and no b ions should score ln(1 * 41) + ln(2!) = {want:.6f}; "
        f"got {hyperscore(y_only, 'GAS', 0.01)!r}"
    )
    print("exercise 5 (hyperscore) looks right")

In [ ]:
_try("exercise 5", lambda: (_check_match_peaks(), _check_hyperscore()))

Why the factorial. ln(*n*!) grows faster than *n*, so each extra match in a series is worth
more than the last. This is computed, not quoted:

In [ ]:
print("matched ions n   ln(n!)")
for _n in (1, 2, 4, 6, 8, 10, 12):
    print(f"{_n:>14d}   {math.lgamma(_n + 1):6.2f}")
print(f"\nsix matches in one series add {math.lgamma(7):.2f}; three in each add "
      f"{2 * math.lgamma(4):.2f}")

## 8. Exercise 6 — `search_spectra()`

The engine itself. For every spectrum:

1. Work out the peptide's neutral mass from the precursor (`neutral_mass`, given).
2. Take the candidates whose mass is within `precursor_ppm` **parts per million of that
   observed mass**, both ends included. The index is sorted by mass, so this is one slice, not
   a scan of every peptide.
3. Score every candidate and keep the best. On an exact tie in score, **a decoy beats a
   target**. Both tie rules here are this lesson's conventions, not a standard: the theory behind
   the estimate assumes ties are absent or broken at random (`claims.yaml`), and this rule errs
   the safe way, since a target that has to share a top score with a decoy cannot be called a
   confident hit. If two candidates of the same kind tie, take the alphabetically first peptide,
   so the result never depends on the order of the index.
4. Report a `PSM` (peptide-spectrum match): scan, peptide, score, whether it is a decoy, and how
   many candidates competed. A spectrum with no candidate in the window produces no PSM.

The `scorer` argument lets the caller swap in another scoring function with the same signature
as `hyperscore`; when it is `None`, use `hyperscore`.

<details><summary>💡 Hint 1 — what to think about</summary>

What is the width of the window, and is it the same on both sides? Which mass is it a fraction
of? Then think about what `np.searchsorted` returns for the left and right ends of a window, and
which `side=` makes each end inclusive. What should happen for a spectrum whose window is empty?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Compute the half-width as the observed mass times `precursor_ppm` over a million, look up the
two slice ends with `np.searchsorted` on `index.masses`, loop over the slice, and keep the best by
the three-part rule: higher score, then decoy over target, then earlier in the alphabet. Skip the
spectrum when the slice is empty. Raise `ValueError` for a negative tolerance of either kind.
</details>

In [ ]:
class PSM(NamedTuple):
    """The best-scoring candidate for one spectrum."""
    scan: int
    peptide: str
    score: float
    is_decoy: bool
    n_candidates: int


def search_spectra(spectra: list, index: PeptideIndex, precursor_ppm: float,
                   fragment_tol: float, scorer: Callable | None = None) -> list:
    """Best candidate peptide for each spectrum, by precursor window then fragment score.

    Candidates for a spectrum are the peptides in `index` with mass in
    `[m - m * ppm / 1e6, m + m * ppm / 1e6]`, ends included, where `m = neutral_mass(spectrum)`.
    The best is the highest `scorer(spectrum, peptide, fragment_tol)`; on an exact tie a decoy
    beats a target, and between two of the same kind the alphabetically first peptide wins.
    `scorer=None` means `hyperscore`. A spectrum with no candidate yields no PSM. The output is in
    the order of `spectra`. Raise `ValueError` if either tolerance is negative.

    Example:
        >>> idx = PeptideIndex(np.array([1000.0, 1000.001]), ("AAAAAAK", "CCCCCCK"), np.array([False, True]))
        >>> spec = Spectrum(7, 1001.0072764666, 1, np.array([100.0]), np.array([100.0]))
        >>> search_spectra([spec], idx, 20.0, 0.5, scorer=lambda s, p, t: 1.0)
        [PSM(scan=7, peptide='CCCCCCK', score=1.0, is_decoy=True, n_candidates=2)]

    Returns:
        A list of `PSM`, one per spectrum that had at least one candidate.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_search_spectra() -> None:
    # An index with made-up masses, so this check does not depend on your other exercises.
    idx = PeptideIndex(np.array([500.0, 1000.0, 1000.001, 1000.5, 2000.0]),
                       ("AAAAAAK", "BBBBBBK", "CCCCCCK", "DDDDDDK", "EEEEEEK"),
                       np.array([False, False, True, False, False]))

    def spec(scan, mass):
        return Spectrum(scan, mass + PROTON, 1, np.array([100.0]), np.array([100.0]))

    marks = {"BBBBBBK": 5.0, "CCCCCCK": 5.0, "DDDDDDK": 9.0, "AAAAAAK": 1.0, "EEEEEEK": 1.0}
    scorer = lambda s, p, t: marks[p]   # noqa: E731
    got = search_spectra([spec(1, 1000.0)], idx, 20.0, 0.5, scorer=scorer)
    assert len(got) == 1 and isinstance(got[0], PSM), "one PSM for a spectrum with candidates"
    assert (got[0].peptide, got[0].is_decoy, got[0].n_candidates) == ("CCCCCCK", True, 2), (
        f"at 20 ppm the window around 1000.0 holds BBBBBBK (1000.0) and CCCCCCK (1000.001) but "
        f"not DDDDDDK (1000.5); they tie on score and the DECOY wins a tie. Got {got[0]!r}"
    )
    wide = search_spectra([spec(2, 1000.0)], idx, 1000.0, 0.5, scorer=scorer)
    assert (wide[0].peptide, wide[0].n_candidates) == ("DDDDDDK", 3), (
        f"at 1000 ppm the window reaches 1000.5, so DDDDDDK (score 9) competes: 3 candidates and "
        f"DDDDDDK the winner; got {wide[0]!r}"
    )
    above = search_spectra([spec(3, 1000.002)], idx, 4.0, 0.5, scorer=scorer)
    under = search_spectra([spec(5, 999.998)], idx, 4.0, 0.5, scorer=scorer)
    assert len(above) == 1 and above[0].n_candidates == 2 and len(under) == 1 \
        and under[0].n_candidates == 2, (
        "the window is symmetric: a spectrum just ABOVE both candidates (1000.002) and one just "
        "BELOW them (999.998) must each reach both, at 4 ppm; got "
        f"{[p.n_candidates for p in above + under]} candidates"
    )
    assert search_spectra([spec(4, 1500.0)], idx, 20.0, 0.5, scorer=scorer) == [], (
        "a spectrum with no candidate in its window gets no PSM"
    )
    ordered = search_spectra([spec(9, 500.0), spec(8, 2000.0)], idx, 20.0, 0.5, scorer=scorer)
    assert [p.scan for p in ordered] == [9, 8], "keep the order of the input spectra"
    first = PeptideIndex(np.array([1000.0, 1000.001]), ("DDDDDDK", "AAAAAAK"), np.array([False, False]))
    got = search_spectra([spec(6, 1000.0)], first, 20.0, 0.5, scorer=scorer)
    assert [(p.peptide, p.n_candidates) for p in got] == [("DDDDDDK", 2)], (
        f"the best candidate can come first in the window: keep it, and still step through every "
        f"candidate after it; got {got!r}"
    )
    print("exercise 6 looks right — candidate window, best score, and the tie rule")

In [ ]:
_try("exercise 6", _check_search_spectra)

Run it on the whole sample: every spectrum against the whole search space, with your
`match_peaks` and `hyperscore` doing the scoring. Then look at where the best hits landed, and at
the assumption the next section rests on.

In [ ]:
def _run_the_search() -> None:
    psms = search_spectra(SPECTRA, _PIPE["index"], PRECURSOR_PPM, FRAGMENT_TOL)
    _PIPE["psms"] = psms
    on_decoy = sum(p.is_decoy for p in psms)
    print(f"{len(SPECTRA)} spectra searched; {len(psms)} had at least one candidate in the "
          f"precursor window (on average {sum(p.n_candidates for p in psms) / len(psms):.1f} of them)")
    print(f"best hits: {len(psms) - on_decoy} on targets, {on_decoy} on decoys")
    scores_t = [p.score for p in psms if not p.is_decoy]
    scores_d = [p.score for p in psms if p.is_decoy]
    fig, ax = plt.subplots(figsize=(8, 2.8))
    bins = np.linspace(0, max(scores_t + scores_d), 45)
    ax.hist(scores_t, bins=bins, alpha=0.6, label="best hit is a target")
    ax.hist(scores_d, bins=bins, alpha=0.6, label="best hit is a decoy")
    ax.set_xlabel("hyperscore of the best hit")
    ax.set_ylabel("spectra")
    ax.legend(fontsize=8)
    fig.tight_layout()
    _show(fig)


_try("search", _run_the_search, needs=_UPTO_6 + ("search space",))

The decoy hits are the low-scoring mass of wrong answers, and the target hits are those plus a
high-scoring hump of right ones. That picture is the whole idea. Its assumption can be tested,
here, because the simulator knows which hits are wrong: **of the wrong best hits, is the share that
landed on decoys about the share of the search space that decoys make up?** If wrong hits
prefer targets, the decoy count understates the wrong-target count, and every FDR estimate built
on it is too kind.

In [ ]:
def _wrong(psm: PSM) -> bool:
    """A best hit is wrong unless it is a target peptide that produced this spectrum."""
    truth = TRUTH[psm.scan - 1]
    return psm.is_decoy or truth is None or not _same_peptide(psm.peptide, truth)


def _test_the_assumption() -> None:
    psms, index = _PIPE["psms"], _PIPE["index"]
    wrong = [p for p in psms if _wrong(p)]
    on_decoy = sum(p.is_decoy for p in wrong)
    share, space_share = on_decoy / len(wrong), float(index.is_decoy.mean())
    z = (share - space_share) / math.sqrt(space_share * (1 - space_share) / len(wrong))
    print(f"wrong best hits:            {len(wrong)}")
    print(f"  landed on decoys:         {on_decoy} = {100 * share:.1f}%")
    print(f"decoys' share of the space: {100 * space_share:.1f}%")
    print(f"the gap is {z:+.1f} standard errors of what {len(wrong)} wrong hits would scatter by "
          "if each landed on a decoy with probability equal to that share")
    print("the target-decoy estimate needs the two shares to agree: "
          + ("they do, to within two standard errors." if abs(z) < 2 else
             "they do not, to within two standard errors, so distrust every estimate below."))


_try("target-decoy assumption", _test_the_assumption, needs=_UPTO_6 + ("search",))

## 9. Exercise 7 — target-decoy q-values

Sort the best hits by score, best first. At any score threshold, *T* target hits and *D* decoy
hits lie at or above it. Every decoy hit is, by construction, a wrong answer, and under the
assumption just tested a wrong target hit is about as likely as a decoy hit, so about *D* of the
*T* target hits above the line are wrong. The estimate of the false discovery rate at that
threshold is **FDR = D / T**. That is the form for a list that reports only the target hits, as
this one does; Elias and Gygi's own estimate is over the combined list of targets and decoys,
2*D* / (*T* + *D*). Both forms are quoted in `claims.yaml`.

Three details, all graded:

- **Ties enter together.** Hits with exactly the same score cannot be split by any threshold, so
  the ratio is evaluated after *all* of a tied group has been counted, and every member of the
  group gets the same value.
- **The q-value of a hit is the smallest FDR at which it would be accepted.** Loosening the
  threshold can lower the raw ratio (a run of good targets), so a hit's q-value is the minimum
  of the raw FDR over its own score *and every lower score*. The result never rises as the score
  falls. When *T* is zero the ratio is undefined and is taken to be 1.0, and no value exceeds
  1.0: two conventions of this lesson.
- **The +1 correction.** Levitsky and colleagues found that when decoy counts are used for
  q-values and filtering, the common equation needs a correction, most of all for small lists and
  very low thresholds (`claims.yaml`). `plus_one=True` applies its "+1" form, *D* replaced by
  *D + 1*. Offering it as an option, with the plain ratio as the default, is this lesson's choice.

A second small function turns q-values into a list: `accepted` marks the **targets** whose
q-value is at most the threshold, the threshold itself included; a decoy is never accepted.

<details><summary>💡 Hint 1 — what to think about</summary>

Sort so that the best score comes first, and take running totals of targets and of decoys. Where
in a run of equal scores is the ratio the right one to use for every member of the run? Then:
if you walk from the worst score to the best, what do you have to carry along so that the q-value
never rises going down the list? Do not forget to put the answers back in the caller's order.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Sort by descending score (stable), form the two running counts, and compute the ratio only at the
last row of each run of equal scores. Take a running minimum from the bottom of the sorted list
upwards, spread each run's value over its members, and scatter the result back by the inverse
of your sort. Raise `ValueError` if the two inputs differ in length, and for an `alpha` outside 0 to 1.
</details>

In [ ]:
def q_values(scores, is_decoy, plus_one: bool = False) -> np.ndarray:
    """Target-decoy q-values, aligned with the input order.

    For each distinct score s, from highest to lowest, count the targets T and decoys D with score
    >= s (a tied group is counted whole). The raw FDR at s is D / T, or (D + 1) / T when
    `plus_one`, or 1.0 if T is 0, never above 1.0. The q-value at s is the minimum raw FDR over s
    and every LOWER score. Every hit, decoys included, receives the q-value of its score.
    Raise `ValueError` if `scores` and `is_decoy` differ in length.

    Example:
        >>> q_values([9.0, 8.0, 7.0, 7.0], [False, False, False, True]).tolist()
        [0.0, 0.0, 0.3333333333333333, 0.3333333333333333]

    Returns:
        A float array with one q-value per input hit.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def accepted(q, is_decoy, alpha: float) -> np.ndarray:
    """Boolean mask of the TARGET hits with q-value <= alpha (the threshold itself is included).

    Decoys are never accepted. Raise `ValueError` if `alpha` is outside [0, 1]: one percent is
    0.01, not 1.

    Example:
        >>> accepted([0.0, 0.01, 0.01, 0.2], [False, False, True, False], 0.01).tolist()
        [True, True, False, False]

    Returns:
        A bool array, aligned with the input.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_q_values() -> None:
    q = q_values([9.0, 8.0, 7.0, 7.0], [False, False, False, True])
    assert np.allclose(q, [0.0, 0.0, 1 / 3, 1 / 3]), (
        f"expected [0, 0, 1/3, 1/3], got {np.round(q, 4).tolist()}. The two hits scored 7 are one "
        "tied group (a target and a decoy): the ratio is taken after BOTH are counted, so the "
        "target is not accepted at a ratio that ignores the decoy tied with it"
    )
    q = q_values([5.0, 4.0, 3.0, 2.0], [False, True, False, False])
    assert np.allclose(q, [0.0, 1 / 3, 1 / 3, 1 / 3]), (
        f"the raw ratios going down are 0, 1, 1/2, 1/3; the q-value of every hit is the smallest raw "
        f"ratio at or below its own score, so [0, 1/3, 1/3, 1/3]; got {np.round(q, 4).tolist()}. "
        "Without the running minimum the q-values are not monotone"
    )
    q1 = q_values([9.0, 8.0], [False, False], plus_one=True)
    assert np.allclose(q1, [0.5, 0.5]), (
        f"with plus_one the ratio is (D + 1) / T = 1/2 at both thresholds, got {np.round(q1, 4).tolist()}"
    )
    assert q_values([1.0, 2.0], [True, True]).tolist() == [1.0, 1.0], (
        "with no targets at all the ratio is undefined: it is 1.0, never a division error or NaN"
    )
    assert len(q_values([], [])) == 0, "empty in, empty out"
    m = accepted([0.0, 0.01, 0.01, 0.2], [False, False, True, False], 0.01)
    assert m.tolist() == [True, True, False, False], (
        f"accepted() is q <= alpha (equal counts) for targets only; got {m.tolist()}"
    )
    for bad in (-0.1, 1.5):
        try:
            accepted([0.0], [False], bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"alpha={bad} must raise ValueError")
    print("exercise 7 looks right — ties, the running minimum, the +1 correction, the boundary")

In [ ]:
_try("exercise 7", _check_q_values)

Now the test that matters, and the one only a simulator can run: **is the estimate honest?** The
table compares, for each threshold, the true proportion of wrong hits among the accepted target
PSMs (the simulator knows which are wrong) with the estimate you were promised.

In [ ]:
ALPHAS = (0.001, 0.005, 0.01, 0.02, 0.05, 0.1)


def _psm_level() -> None:
    psms = _PIPE["psms"]
    scores = np.array([p.score for p in psms])
    decoy = np.array([p.is_decoy for p in psms])
    wrong = np.array([_wrong(p) for p in psms])
    q, q_plus = q_values(scores, decoy), q_values(scores, decoy, plus_one=True)
    print(f"{'promised':>9s} {'accepted':>9s} {'wrong':>6s} {'true share':>11s} "
          f"{'accepted, +1 rule':>18s}")
    for alpha in ALPHAS:
        keep = accepted(q, decoy, alpha)
        n, bad = int(keep.sum()), int((keep & wrong).sum())
        print(f"{alpha:9.3f} {n:9d} {bad:6d} {100 * bad / max(n, 1):10.2f}% "
              f"{int(accepted(q_plus, decoy, alpha).sum()):18d}")
    print("\nEach row: accept every target PSM whose q-value is at most the promised rate. The true share")
    print("is what the simulator's answer key says. Where a row's share is above what was promised, look")
    print("at how few wrong PSMs are involved: a promise about a rate is kept on average, not in every list.")


_try("PSM-level check", _psm_level, needs=_UPTO_7 + ("search",))

## 10. Exercise 8 — from PSMs to peptides

The same peptide can be seen in several spectra. Counting every PSM as a separate finding
overstates what was found, and the error rate of the *peptide list* need not be that of the
*PSM list*: a peptide seen twice counts twice among the PSMs and once among the peptides. The
cell after this exercise prints both, and how often the right and the wrong peptides were seen.

`best_per_peptide` collapses the PSM list to one entry per distinct peptide sequence: its best
score, whether it is a decoy, and how many PSMs it had. FDR is then estimated **again**, on
peptides, with the same `q_values`.

<details><summary>💡 Hint 1 — what to think about</summary>

Which of a peptide's several PSMs should represent it, and what does that choice do to a wrong
peptide seen once next to a right peptide seen twenty times? What should the count of PSMs be?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Group by peptide sequence, keep the highest score in each group and the number of PSMs in it.
Return the peptides in the order they first appear in the input.
</details>

In [ ]:
class PeptideHit(NamedTuple):
    """One distinct peptide, at the score of its best PSM."""
    peptide: str
    score: float
    is_decoy: bool
    n_psms: int


def best_per_peptide(psms: list) -> list:
    """Collapse PSMs to one hit per distinct peptide: its best score and its PSM count.

    Output is in the order each peptide first appears in `psms`.

    Example:
        >>> a = PSM(1, "AAAAAAK", 5.0, False, 3)
        >>> b = PSM(2, "AAAAAAK", 9.0, False, 2)
        >>> c = PSM(3, "KAAAAAA", 4.0, True, 2)
        >>> best_per_peptide([c, a, b])
        [PeptideHit(peptide='KAAAAAA', score=4.0, is_decoy=True, n_psms=1), PeptideHit(peptide='AAAAAAK', score=9.0, is_decoy=False, n_psms=2)]

    Returns:
        A list of `PeptideHit`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_best_per_peptide() -> None:
    a = PSM(1, "AAAAAAK", 5.0, False, 3)
    b = PSM(2, "AAAAAAK", 9.0, False, 2)
    c = PSM(3, "KAAAAAA", 4.0, True, 2)
    d = PSM(4, "AAAAAAK", 7.0, False, 1)
    got = best_per_peptide([c, a, b, d])
    assert got == [PeptideHit("KAAAAAA", 4.0, True, 1), PeptideHit("AAAAAAK", 9.0, False, 3)], (
        f"expected KAAAAAA once, then AAAAAAK at its BEST score 9.0 over 3 PSMs, in order of first "
        f"appearance (not sorted); got {got!r}. The score is the best one, not the first or the last"
    )
    assert best_per_peptide([]) == [], "no PSMs, no peptides"
    print("exercise 8 looks right — one hit per peptide, at its best score")

In [ ]:
_try("exercise 8", _check_best_per_peptide)

In [ ]:
def _true_peptides() -> set:
    return {t.replace("I", "L") for t in TRUTH if t is not None}


def _peptide_level() -> None:
    hits = best_per_peptide(_PIPE["psms"])
    scores = np.array([h.score for h in hits])
    decoy = np.array([h.is_decoy for h in hits])
    q = q_values(scores, decoy)
    _PIPE.update(hits=hits, pep_q=q)
    real = _true_peptides()
    wrong = np.array([h.is_decoy or h.peptide.replace("I", "L") not in real for h in hits])
    psm_scores = np.array([p.score for p in _PIPE["psms"]])
    psm_decoy = np.array([p.is_decoy for p in _PIPE["psms"]])
    psm_keep = accepted(q_values(psm_scores, psm_decoy), psm_decoy, ALPHA)
    keep = accepted(q, decoy, ALPHA)
    print(f"{len(_PIPE['psms'])} PSMs collapse to {len(hits)} distinct peptides "
          f"(each seen {sum(h.n_psms for h in hits) / len(hits):.2f} times on average)")
    print(f"at a promised rate of {ALPHA}: {int(psm_keep.sum())} PSMs accepted, but "
          f"{int(keep.sum())} peptides accepted when FDR is estimated on peptides")
    print(f"of the peptides accepted, {int((keep & wrong).sum())} are not peptides of the sample "
          f"({100 * (keep & wrong).sum() / max(keep.sum(), 1):.2f}%)")
    seen_right = [h.n_psms for h, k, w in zip(hits, keep, wrong) if k and not w]
    seen_wrong = [h.n_psms for h, k, w in zip(hits, keep, wrong) if k and w]
    if seen_right and seen_wrong:
        print(f"peptides of the sample were seen in {np.mean(seen_right):.2f} spectra on average; "
              f"the wrong ones, in {np.mean(seen_wrong):.2f}")


_try("peptide-level check", _peptide_level, needs=_UPTO_8 + ("search",))

## 11. Exercise 9 — protein inference by parsimony

You have accepted a list of peptides. The question you were asked was about proteins. A peptide
that occurs in only one database protein points at it; a peptide shared between proteins does
not say which was in the sample. Listing every protein that contains any accepted peptide is
the *naive* answer, and it inflates the count: a truncated protein whose peptides are all
shared with its parent is "found" by evidence that the parent already explains.

The **parsimony** answer (the principle is sourced in `claims.yaml`) is the smallest set of
proteins that explains every accepted peptide. `protein_groups` takes `evidence`, a dict
mapping each accepted peptide to the proteins that contain it, and returns groups of proteins:

1. Proteins with **exactly the same** accepted peptides cannot be told apart: they form one
   *group* (a tuple of ids, sorted).
2. A group whose peptides are a **proper subset** of another group's is dropped: the other
   group already explains all of it.
3. Of what remains, repeatedly take the group that explains the most peptides not yet
   explained (on a tie, the group whose tuple sorts first), until every peptide is explained.
   A group that would add nothing is not taken.

Step 3 is greedy: it finds a small set that explains everything, not a guaranteed smallest one.
The result is a list of groups, **sorted**.

<details><summary>💡 Hint 1 — what to think about</summary>

Turn the map around: for each protein, which accepted peptides does it hold? Two proteins with
the same set are one thing. Then ask, of two groups where one holds strictly more than the
other, which is the parsimonious one? Why must that step come before the greedy step, and not
be left to it? Consider what greedy does when two groups tie on what they add.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Build `peptides_of[protein]`, group proteins by that set, remove groups whose set is a proper
subset of another group's, then loop: pick the remaining group covering the most unexplained
peptides (tie: smaller group tuple), record it, and remove its peptides from the unexplained set.
Recomputing every group's cover on each pass is fine here; keep the whole thing correct first.
</details>

In [ ]:
def protein_groups(evidence: dict) -> list:
    """The smallest sensible set of protein groups that explains every peptide in `evidence`.

    `evidence` maps each accepted peptide to a tuple of the proteins that contain it. Proteins with
    identical peptide sets form one group (a sorted tuple of ids). A group whose peptides are a
    proper subset of another group's is dropped. Then, greedily, take the group that explains the
    most still-unexplained peptides (ties: the smaller group tuple), until none adds anything.

    Example:
        >>> protein_groups({"p1": ("A", "B"), "p2": ("A",), "p3": ("C",)})
        [('A',), ('C',)]
        >>> protein_groups({"p1": ("A", "B"), "p2": ("A", "B")})
        [('A', 'B')]

    Returns:
        A sorted list of groups, each a tuple of protein ids.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_protein_groups() -> None:
    got = protein_groups({"p1": ("A", "B"), "p2": ("A",), "p3": ("C",)})
    assert got == [("A",), ("C",)], (
        f"expected [('A',), ('C',)], got {got!r}. B holds only p1, which A also holds: A explains "
        "everything B could, so B is not needed"
    )
    got = protein_groups({"p1": ("A", "B"), "p2": ("A", "B")})
    assert got == [("A", "B")], (
        f"A and B hold exactly the same peptides and cannot be told apart: one group ('A', 'B'), "
        f"got {got!r}"
    )
    got = protein_groups({"p1": ("X", "Z"), "p2": ("X", "Y"), "p3": ("Y", "Z")})
    assert got == [("X",), ("Y",)], (
        f"X, Y and Z each hold two of three peptides and any two explain everything; taking "
        f"the smallest tuple on each tie gives [('X',), ('Y',)], got {got!r}"
    )
    got = protein_groups({"p2": ("A", "B", "X"), "p3": ("B", "X"), "p5": ("X",), "p6": ("X",),
                          "p1": ("A", "B")})
    assert got == [("B",), ("X",)], (
        f"A holds {{p1, p2}}, B holds {{p1, p2, p3}}, X holds {{p2, p3, p5, p6}}. A is a proper subset "
        f"of B, so A goes even though it would win a tie on its name: expected [('B',), ('X',)], "
        f"got {got!r}"
    )
    assert protein_groups({}) == [], "no evidence, no proteins"
    print("exercise 9 looks right — groups, subsets, and the greedy cover")

In [ ]:
_try("exercise 9", _check_protein_groups)

Naive against parsimonious, on the peptides accepted at the promised rate. Both counts are
computed from your code, and they are counts of *reported proteins*, not of anything true yet.

In [ ]:
def _accepted_evidence(alpha: float) -> dict:
    """Accepted peptides -> the proteins that contain them."""
    hits, q, space = _PIPE["hits"], _PIPE["pep_q"], _PIPE["space"]
    decoy = np.array([h.is_decoy for h in hits])
    keep = accepted(q, decoy, alpha)
    return {h.peptide: space[h.peptide] for h, k in zip(hits, keep) if k}


def _naive_against_parsimony() -> None:
    evidence = _accepted_evidence(ALPHA)
    naive = {pid for ids in evidence.values() for pid in ids}
    groups = protein_groups(evidence)
    print(f"{len(evidence)} peptides accepted at a promised rate of {ALPHA}")
    print(f"  every protein that contains one of them:  {len(naive)}")
    print(f"  smallest set of groups that explains them: {len(groups)}")
    print(f"  of those groups, {sum(len(g) > 1 for g in groups)} hold several proteins that "
          "the evidence cannot tell apart")


_try("naive against parsimony", _naive_against_parsimony, needs=_UPTO_9 + ("peptide-level check",))

## 12. Exercise 10 — protein-level scores and protein-level q-values

Everything so far controlled error on *peptides*. But the list you publish is a list of
proteins, and section 13 shows what happens if you only ever controlled the peptides. Error has
to be estimated again, on the thing you report, in the same way: score the protein groups, mark
the decoy ones, and run the target-decoy q-values on them.

`protein_results` takes the groups from `protein_groups`, the same `evidence` and the best
score of each peptide, and returns one `ProteinResult` per group, in the order of `groups`:

- **score**: the best peptide score among the peptides the group explains (those in `evidence`
  that any of its proteins holds). The best, not the sum: a sum rewards long proteins, which
  have more peptides to be wrong on.
- **n_peptides**: how many peptides that is.
- **is_decoy**: every protein in the group carries `DECOY_PREFIX`.
- **q**: from `qfn(scores, is_decoy)` over all the groups at once; `qfn=None` means `q_values`.

<details><summary>💡 Hint 1 — what to think about</summary>

For each group you need the peptides it accounts for, which means going from a protein back to
its peptides. A peptide shared by two groups counts for both. What score should a group with no
peptide at all get, and can that ever happen given where the groups came from?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Invert `evidence` to a map from protein to the peptides it holds, take the union over the members
of each group, and read the score and the count off that union. Collect all the scores and decoy
flags first, call the q-value function once on the lot, and only then build the result list.
</details>

In [ ]:
class ProteinResult(NamedTuple):
    """One protein group with its evidence, its score and its q-value."""
    group: tuple
    score: float
    n_peptides: int
    is_decoy: bool
    q: float


def protein_results(groups: list, evidence: dict, peptide_scores: dict,
                    qfn: Callable | None = None) -> list:
    """Score each protein group, mark decoy groups, and attach protein-level q-values.

    A group's peptides are the peptides in `evidence` held by any of its proteins. Its score is
    the largest `peptide_scores[peptide]` among them (0.0 if it has none); `n_peptides` counts
    them. A group is a decoy if every protein id in it starts with `DECOY_PREFIX`. `q` comes from
    one call `qfn(scores, is_decoy)` over all the groups (`None` means `q_values`). The output is
    in the order of `groups`.

    Example:
        >>> ev = {"p1": ("A",), "p2": ("A", "C"), "p3": ("DECOY_B",)}
        >>> r = protein_results([("A",), ("DECOY_B",)], ev, {"p1": 5.0, "p2": 9.0, "p3": 4.0})
        >>> [(x.group, x.score, x.n_peptides, x.is_decoy) for x in r]
        [(('A',), 9.0, 2, False), (('DECOY_B',), 4.0, 1, True)]

    Returns:
        A list of `ProteinResult`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_protein_results() -> None:
    ev = {"p1": ("A",), "p2": ("A", "C", "D"), "p3": ("DECOY_B",), "p4": ("C", "D")}
    scores = {"p1": 5.0, "p2": 9.0, "p3": 4.0, "p4": 3.0}
    groups = [("A",), ("DECOY_B",), ("C", "D")]
    got = protein_results(groups, ev, scores, qfn=lambda s, d: np.arange(len(s)) / 10.0)
    assert [r.group for r in got] == groups, "one result per group, in the order of `groups`"
    assert [r.score for r in got] == [9.0, 4.0, 9.0], (
        f"a group's score is the BEST score among the peptides it explains (A: p1 5.0, p2 9.0 -> "
        f"9.0), got {[r.score for r in got]!r}. A shared peptide counts for every group holding it"
    )
    assert [r.n_peptides for r in got] == [2, 1, 2], (
        f"counts of explained peptides should be [2, 1, 2], got {[r.n_peptides for r in got]!r}"
    )
    assert [r.is_decoy for r in got] == [False, True, False], (
        "a group is a decoy when EVERY protein in it carries the DECOY_ prefix"
    )
    assert [r.q for r in got] == [0.0, 0.1, 0.2], (
        f"q-values must come from qfn(scores, is_decoy) over all groups together, in order; "
        f"got {[r.q for r in got]!r}"
    )
    default = protein_results(groups, ev, scores)
    assert np.allclose([r.q for r in default], [0.0, 0.5, 0.0]), (
        f"with the default q_values the tied targets at 9.0 are one group (ratio 0) and the decoy "
        f"at 4.0 sees 1 decoy over 2 targets = 0.5; got {[r.q for r in default]!r}"
    )
    assert protein_results([], ev, scores) == [], "no groups, no results"
    print("exercise 10 looks right — protein scores, decoy groups, and protein-level q-values")

In [ ]:
_try("exercise 10", _check_protein_results)

## 13. The result: one threshold, two levels, and an answer key

Everything is in place. The cell below takes the same PSMs, the same peptides and the same
promised rate, and produces three protein lists:

- **peptide-level control**: accept peptides at the promised rate (section 10), then infer
  proteins from the accepted peptides (section 11);
- **the same, plus the two-peptide rule**: keep only groups with at least two accepted peptides,
  a folk remedy for one-hit wonders;
- **protein-level control**: infer proteins from *all* peptides, score the groups, and accept
  groups at the promised rate on protein q-values (section 12).

A reported group is **absent** when none of its proteins is in the sample. Read the table before
the paragraph after it.

In [ ]:
def _true_group(group: tuple) -> bool:
    return any(pid in PRESENT for pid in group)


def _the_comparison() -> None:
    hits, space = _PIPE["hits"], _PIPE["space"]
    evidence = {h.peptide: space[h.peptide] for h in hits}
    peptide_score = {h.peptide: h.score for h in hits}
    groups = protein_groups(evidence)
    results = protein_results(groups, evidence, peptide_score)
    _PIPE["results"] = results
    print(f"{len(hits)} peptides -> {len(groups)} protein groups, decoys included, before any "
          "threshold\n")
    rows = []
    for alpha in ALPHAS:
        ev = _accepted_evidence(alpha)
        by_peptides = protein_groups(ev)
        two = [g for g in by_peptides
               if sum(any(pid in ids for pid in g) for ids in ev.values()) >= 2]
        by_proteins = [r.group for r in results if not r.is_decoy and r.q <= alpha]
        rows.append((alpha, len(ev)) + tuple(
            (len(lst), sum(not _true_group(g) for g in lst))
            for lst in (by_peptides, two, by_proteins)))
    _PIPE["rows"] = rows
    print(f"{'promised':>8s} {'peptides':>8s} | {'peptide-level':^21s} | "
          f"{'+ two-peptide rule':^21s} | {'protein-level':^21s}")
    print(f"{'':>8s} {'':>8s} | " + " | ".join(
        f"{'proteins':>8s} {'absent':>6s} {'%':>5s}" for _ in range(3)))
    for alpha, n_pep, *levels in rows:
        cells = " | ".join(f"{n:8d} {bad:6d} {100 * bad / max(n, 1):5.1f}" for n, bad in levels)
        print(f"{alpha:8.3f} {n_pep:8d} | {cells}")


_try("the comparison", _the_comparison, needs=_UPTO_10 + ("peptide-level check",))

In [ ]:
def _plot_the_comparison() -> None:
    rows = _PIPE["rows"]
    alphas = [r[0] for r in rows]
    fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.4))
    for k, (label, style) in enumerate((("peptide-level control", "o-"),
                                        ("+ two-peptide rule", "s--"),
                                        ("protein-level control", "^-"))):
        left.plot(alphas, [r[2 + k][0] for r in rows], style, label=label)
        right.plot(alphas, [100 * r[2 + k][1] / max(r[2 + k][0], 1) for r in rows], style,
                   label=label)
    right.plot(alphas, [100 * a for a in alphas], "k:", label="what was promised")
    left.set_xscale("log")
    right.set_xscale("log")
    left.set_xlabel("promised rate")
    right.set_xlabel("promised rate")
    left.set_ylabel("proteins reported")
    right.set_ylabel("reported proteins absent from the sample, %")
    right.legend(fontsize=8)
    fig.tight_layout()
    _show(fig)


_try("the comparison plot", _plot_the_comparison, needs=_UPTO_10 + ("the comparison",))

In [ ]:
def _the_headline() -> None:
    row = next(r for r in _PIPE["rows"] if r[0] == ALPHA)
    (n1, b1), (n2, b2), (n3, b3) = row[2:]
    print(f"at a promised rate of {ALPHA}:")
    print(f"  controlling error on peptides:   {n1} proteins reported, {b1} absent from the "
          f"sample ({100 * b1 / max(n1, 1):.1f}%)")
    print(f"  ... plus the two-peptide rule:   {n2} proteins reported, {b2} absent "
          f"({100 * b2 / max(n2, 1):.1f}%)")
    print(f"  controlling error on proteins:   {n3} proteins reported, {b3} absent "
          f"({100 * b3 / max(n3, 1):.1f}%)")
    share1, share3 = b1 / max(n1, 1), b3 / max(n3, 1)
    print(f"\nthe peptide list was accepted at the promised rate; the protein list built from it "
          f"{'was above it' if share1 > ALPHA else 'was not above it'}: "
          f"{100 * share1:.1f}% absent against {100 * ALPHA:.1f}% promised.")
    print(f"protein-level control reports {n3 - n1:+d} proteins relative to it, and "
          f"{100 * share3:.1f}% of its list is absent, "
          f"{'above' if share3 > ALPHA else 'not above'} the promised {100 * ALPHA:.1f}%.")


_try("headline", _the_headline, needs=_UPTO_10 + ("the comparison",))

In [ ]:
def _what_holds_a_protein_up() -> None:
    """For the peptide-level protein list: how many accepted peptides stand behind each group."""
    space = _PIPE["space"]
    real = _true_peptides()
    evidence = _accepted_evidence(ALPHA)
    groups = protein_groups(evidence)
    wrong = [p for p in evidence if p.replace("I", "L") not in real]
    support = {g: sum(any(pid in space[p] for pid in g) for p in evidence) for g in groups}
    absent = [n for g, n in support.items() if not _true_group(g)]
    present = [n for g, n in support.items() if _true_group(g)]
    print(f"peptide-level list at {ALPHA}: {len(evidence)} accepted peptides, {len(wrong)} of them "
          f"not peptides of the sample, roll up to {len(groups)} protein groups")
    if absent and present:
        print(f"  the {len(absent)} groups absent from the sample stand on {np.mean(absent):.2f} "
              f"accepted peptides each on average; {sum(n == 1 for n in absent)} of them on exactly one")
        print(f"  the {len(present)} groups present in the sample stand on {np.mean(present):.2f} on "
              f"average; {sum(n == 1 for n in present)} of them on exactly one")


_try("what holds a protein up", _what_holds_a_protein_up, needs=_UPTO_10 + ("peptide-level check",))

In [ ]:
def _absent_but_supported() -> None:
    """Which absent proteins survive protein-level control, and what holds them up."""
    hits, space, results = _PIPE["hits"], _PIPE["space"], _PIPE["results"]
    real = _true_peptides()
    absent = [r for r in results if not r.is_decoy and r.q <= ALPHA and not _true_group(r.group)]
    print(f"{len(absent)} absent protein groups get through protein-level control at {ALPHA}")
    for r in absent:
        mine = [h.peptide for h in hits if any(pid in r.group for pid in space[h.peptide])]
        right = [p for p in mine if p.replace("I", "L") in real]
        both = sum(any(pid in PRESENT for pid in space[p]) for p in right)
        print(f"  {', '.join(r.group):<14s} explains {len(mine)} peptides; {len(right)} of them "
              f"really were in the sample, and {both} of those are also held by a protein in the "
              "sample")
    if absent:
        print("Read the counts: where a group is held up by peptides that really were in the sample, "
              "the evidence is real")
        print("and the address is the problem. No error estimate on peptides or proteins can tell "
              "such a group from a right one.")


_try("absent but supported", _absent_but_supported, needs=_UPTO_10 + ("the comparison",))

Read across the table. Loosening the promised rate makes every list longer. Compare how fast the
absent share grows in each column with how fast the promise grows, and with the dotted line in the
plot. The cell that prints how many accepted peptides stand behind each protein shows the
mechanism at work: nothing makes a wrong peptide arrive together with others from the same
protein, so a wrong peptide that survives tends to be the only evidence for a protein of its own,
while the peptides that are right gather on the proteins that really are in the sample. The
peptide list's error rate is counted per peptide, and the protein list is counted per protein.
On large real data sets Reiter and colleagues found protein error rates significantly above
those of the PSMs (`claims.yaml`).

Three things the table does not settle. Compare the two-peptide column with the other two, on both
counts: what it takes out of the absent column it also takes out of the correct proteins, and it
has no error rate to promise. And the protein-level estimate here is the plain one: Savitski and
colleagues showed that on very large data sets it over-counts decoys, and proposed picking the
better of each target-decoy protein pair (`claims.yaml`). This lesson does not test for that
effect. And protein-level control is no guarantee either: its decoys stand in for peptides that
are wrong, not for right peptides credited to the wrong protein. The last cell above looks at the
absent proteins that get through protein-level control: read what holds each of them up.

## 14. Common mistakes

- **Quoting "1% FDR" without the level.** PSM, peptide and protein FDR are three different
  promises about three different lists, and section 13 puts two of them side by side.
- **Filtering at the PSM or peptide level and calling the protein list "1%".** The promise was
  made about another list; section 13 measures the gap in this world.
- **Breaking ties toward the target.** Every exact tie with a decoy then becomes a confident-
  looking target hit, and the decoy count, and with it the FDR estimate, is too kind.
- **A decoy that is also a target.** A reversed palindrome, or a low-complexity repeat, gives a
  decoy peptide that is a real target peptide. Counting a hit on it as a decoy puts a possibly
  right answer in the wrong column.
- **Not enforcing the running minimum.** Raw D/T rises and falls as the threshold moves; q-values
  that do are not q-values, and a list "at 1%" might contain a hit with a worse estimate than one
  left out.
- **Reporting every protein that contains an accepted peptide.** A truncated form or a paralog
  whose peptides are all shared with another protein is not evidence of itself.
- **Trusting the decoy count without checking the assumption.** Section 8 tested it on a sample
  with an answer key. On a real sample there is none, which is why the assumption has to be tested
  in other ways, and why spectra of peptides missing from the database (variants, modified
  forms) can strain it.
- **Comparing a reported protein count with a published one without the published settings.** The
  methods statement below is the minimum needed to say what changed.

In [ ]:
def _why_the_running_minimum() -> None:
    def raw_ratio(is_decoy: np.ndarray) -> np.ndarray:
        """D / T after each row, best score first, ties ignored: a filter without the minimum."""
        return np.cumsum(is_decoy) / np.maximum(np.cumsum(~is_decoy), 1)

    # A toy list, best hit first: two targets, one decoy, then a long run of targets.
    flags = np.array([False, False, True] + [False] * 10)
    toy_q = q_values(np.arange(len(flags), 0, -1, dtype=float), flags)
    toy_raw = raw_ratio(flags)
    level = 0.15
    print(f"toy list of {len(flags)} hits, one decoy at rank 3, promised rate {level}")
    print("  raw D/T going down: " + " ".join(f"{x:.2f}" for x in toy_raw))
    print("  q-value going down: " + " ".join(f"{x:.2f}" for x in toy_q))
    n_raw, n_q = int(((toy_raw <= level) & ~flags).sum()), int(accepted(toy_q, flags, level).sum())
    print(f"  targets accepted by a filter on the raw ratio: {n_raw}; by a filter on the q-value: {n_q}")

    psms = _PIPE["psms"]
    scores = np.array([p.score for p in psms])
    decoy = np.array([p.is_decoy for p in psms])
    q = q_values(scores, decoy)
    order = np.argsort(-scores, kind="stable")
    raw = raw_ratio(decoy[order])
    print("\nthe same comparison on the PSMs of this world (ties ignored in the raw ratio):")
    print(f"{'promised':>9s} {'raw ratio':>10s} {'q-value':>8s}")
    differ = 0
    for alpha in ALPHAS:
        by_raw = int(((raw <= alpha) & ~decoy[order]).sum())
        by_q = int(accepted(q, decoy, alpha).sum())
        differ += by_raw != by_q
        print(f"{alpha:9.3f} {by_raw:10d} {by_q:8d}")
    print(f"the two filters disagree at {differ} of {len(ALPHAS)} promised rates on these PSMs; on the "
          f"toy list they disagree by {n_q - n_raw} targets")


_try("running minimum", _why_the_running_minimum, needs=_UPTO_7 + ("search",))

## 15. Self-check

1. The best hit for a spectrum is a peptide from a decoy protein. What does that tell you?
   - (a) the sample contained the decoy protein
   - (b) the engine gave a wrong answer, and a wrong answer was as likely to fall on a target
   - (c) the search failed on that spectrum and should be rerun

2. An exact tie in score between a target and a decoy is resolved in favour of the target. The
   estimated FDR is then:
   - (a) unchanged, ties carry no information
   - (b) too high, because decoys are over-counted
   - (c) too low, because decoy hits are counted as target hits

3. Why is a hit's q-value the smallest raw D/T over its own score and every lower one?
   - (a) to make the estimate larger and so safer
   - (b) so that ties are broken in the hit's favour
   - (c) because a looser threshold that accepts everything a stricter one does, at a lower
         ratio, gives the hit a lower FDR at which it is accepted

4. Why can a protein list built from peptides accepted at a given promised rate be wrong more
   often than promised?
   - (a) a wrong peptide is usually the only evidence for its protein, while right peptides
         concentrate on few proteins, so wrong peptides make up a larger share of proteins
   - (b) proteins are longer than peptides, so they score higher by chance
   - (c) the decoy database is too small at the protein level

5. Protein B's only accepted peptide is also in protein A, which has other accepted peptides.
   Under parsimony B is:
   - (a) reported, because its peptide is present
   - (b) not reported separately: A explains the evidence and B adds nothing the data require
   - (c) reported with a lower score than A

Answers, with the reasoning, come with this lesson's worked solution when you enrol on Synapsa.

In [ ]:
def _methods_statement() -> None:
    results = _PIPE["results"]
    kept = [r for r in results if not r.is_decoy and r.q <= ALPHA]
    print("METHODS — a protein count without these lines cannot be reproduced or compared")
    print(f"  enzyme            trypsin, cuts after K or R not before P; up to {MISSED} missed "
          f"cleavage; peptides of {MIN_LEN}-{MAX_LEN} residues")
    print(f"  precursor         {PRECURSOR_PPM:g} ppm of the observed neutral mass; fragments "
          f"{FRAGMENT_TOL:g} Da; singly charged b and y ions")
    print("  score             hyperscore; best candidate per spectrum; a decoy wins an exact tie")
    print("  decoys            every protein reversed, searched concatenated with the targets; a "
          "peptide also found in a target is a target")
    print(f"  error control     target-decoy q-values, plain D/T, at level PROTEIN GROUP, "
          f"promised rate {ALPHA:g}")
    print("  protein score     best peptide score; groups by parsimony (identical peptide sets are "
          "one group,")
    print("                    subset groups dropped, greedy cover)")
    print(f"  reported          {len(kept)} protein groups, of which "
          f"{sum(len(r.group) > 1 for r in kept)} hold several indistinguishable proteins")
    n1, b1 = next(r for r in _PIPE["rows"] if r[0] == ALPHA)[2]
    print(f"  the level matters: controlled at peptide level instead, the same spectra and the same "
          f"promised rate give {n1} proteins, {b1} of them absent from the sample (section 13).")


_try("methods statement", _methods_statement, needs=_UPTO_10 + ("the comparison",))

## What you built, and where it goes next

You now have every layer between a spectrum and a protein count, each of which you wrote and
tested: chemistry, a search, a score, an error estimate, an inference step and an error estimate
again at the level you report. You also have the one thing a real data set never gives you, an
answer key, and you used it to check the estimates against the truth.

The next module in this programme moves from *which* proteins are there to *how much*: the
same spectra, quantified, with the missing values and batch effects that decide whether a
differential-abundance call survives. The habit to carry forward is the one in the methods
statement above: **a protein count is a claim about a list, and the list has a level.**

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")
    stuck = [label for label, state in _STATUS.items() if label not in _EXERCISES and state == "failed"]
    if stuck:
        print("cells that ran your code on the whole world and failed: " + ", ".join(stuck) + ". "
              "A small check cannot see everything the whole world asks; each printed why above.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_digest),
                              ("exercise 2", _check_masses),
                              ("exercise 3", _check_build_peptide_map),
                              ("exercise 4", _check_decoys),
                              ("exercise 5", lambda: (_check_match_peaks(), _check_hyperscore())),
                              ("exercise 6", _check_search_spectra),
                              ("exercise 7", _check_q_values),
                              ("exercise 8", _check_best_per_peptide),
                              ("exercise 9", _check_protein_groups),
                              ("exercise 10", _check_protein_results)):
            _try(_name, _check)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.